# Modelagem

Nesse notebook iremos testar diversas abordagens de modelagem e iremos escolher a que performar melhor no nosso dataset.

## Importando as Bibliotecas

Primeiro, vamos importar nossas bibliotecas.

In [3]:
import numpy as np
import pandas as pd

## Carregando o Dataset

Agora, vamos carregar o dataset.

In [4]:
df = pd.read_parquet("../data/processed/03_processed.parquet")

In [5]:
df = df.astype("float")

In [6]:
df.info()

<class 'pandas.DataFrame'>
Index: 7253 entries, 0 to 9838
Data columns (total 83 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   preco                        7253 non-null   float64
 1   condominio                   5627 non-null   float64
 2   iptu                         4258 non-null   float64
 3   tamanho                      7253 non-null   float64
 4   quartos                      7252 non-null   float64
 5   banheiros                    7252 non-null   float64
 6   vagas_estacionamento         7051 non-null   float64
 7   andar                        3262 non-null   float64
 8   piscina                      7253 non-null   float64
 9   elevador                     7253 non-null   float64
 10  churrasqueira                7253 non-null   float64
 11  condominio_fechado           7253 non-null   float64
 12  academia                     7253 non-null   float64
 13  espaco_gourmet               7253 

## Divisão Treino-Teste

Nessa etapa de modelagem faremos otimização de hiperparâmetros dos nossos modelos e para checarmos a qualidade final do modelo iremos testâ-lo em dados de teste. Para isso precisamos dividir os dados de treinamento e de teste.

In [7]:
from sklearn.model_selection import train_test_split

X = df.drop("preco", axis="columns").values
y = df["preco"].values

random_state = 1667

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.7,
    random_state=random_state,
)

## Regressão Linear com Regularização L1 (Lasso)

O primeiro modelo que iremos treinar é o modelo de Regressão Linear com Regularização L1, também conhecida como Lasso, sendo esse modelo o nosso baseline. Faremos a otimização de hiperparâmetros desse modelo usando a Validação Cruzada com 5 folds e usando a biblioteca `Optuna`.

In [31]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Lasso
from sklearn.model_selection import (
    KFold,
    cross_val_score
)
from sklearn.pipeline import Pipeline

import optuna


kfold = KFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def get_lasso_regression_pipeline(parameters):
    imputer = KNNImputer(
        n_neighbors=parameters["n_neighbors"],
        weights="distance",
    )

    scaler = RobustScaler()

    model = Lasso(
        alpha=parameters["alpha"],
        random_state=random_state,
    )

    pipeline = Pipeline(steps=[
        ("imputer", imputer),
        ("scaler", scaler),
        ("model", model)
    ])

    return pipeline


def objective_lasso_regression(trial):
    pipeline = get_lasso_regression_pipeline({
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 20),
        "alpha": trial.suggest_float("alpha", 0.0, 1000.0),
    })

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=kfold,
        scoring="r2",
    )

    return scores.mean()

Agora que montamos nossa função objetivo, vamos criar nosso estudo e executar a otimização de hiperparâmetros.

In [ ]:
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective_lasso_regression,
    n_trials=100,
    show_progress_bar=True,
)

[I 2026-08-02 06:57:57,231] A new study created in memory with name: no-name-fe5b741e-d371-44da-8287-6aa73a33970e
Best trial: 0. Best value: 0.789819:   2%|▏         | 1/50 [00:08<06:38,  8.13s/it]

[I 2026-08-02 06:58:05,361] Trial 0 finished with value: 0.789818595510131 and parameters: {'n_neighbors': 4, 'alpha': 589.4704193809736}. Best is trial 0 with value: 0.789818595510131.


Best trial: 0. Best value: 0.789819:   4%|▍         | 2/50 [00:16<06:34,  8.21s/it]

[I 2026-08-02 06:58:13,625] Trial 1 finished with value: 0.7860061835993168 and parameters: {'n_neighbors': 20, 'alpha': 290.1521004100329}. Best is trial 0 with value: 0.789818595510131.


Best trial: 0. Best value: 0.789819:   6%|▌         | 3/50 [00:24<06:23,  8.16s/it]

...

[I 2026-08-02 07:04:14,340] Trial 47 finished with value: 0.790361307623396 and parameters: {'n_neighbors': 1, 'alpha': 570.0241239552752}. Best is trial 18 with value: 0.7909031999673678.


Best trial: 18. Best value: 0.790903:  98%|█████████▊| 49/50 [06:25<00:07,  7.92s/it]

[I 2026-08-02 07:04:22,433] Trial 48 finished with value: 0.79018589058073 and parameters: {'n_neighbors': 3, 'alpha': 866.3189802351799}. Best is trial 18 with value: 0.7909031999673678.


Best trial: 18. Best value: 0.790903: 100%|██████████| 50/50 [06:33<00:00,  7.86s/it]

[I 2026-08-02 07:04:30,433] Trial 49 finished with value: 0.7900205040954422 and parameters: {'n_neighbors': 18, 'alpha': 748.906223401332}. Best is trial 18 with value: 0.7909031999673678.
